# 02. XAI Gradient Propagation: Integrated Gradients, LRP, Grad-CAM, and Attention

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Simonyan et al. (2013 *Deep Inside Convolutional Networks*); Bach et al. (2015 *Layer-wise Relevance Propagation*); Shrikumar et al. (2017 *Learning Important Features Through Propagating Activation Differences: DeepLIFT*); Sundararajan et al. (2017 *Axiomatic Attribution for Deep Networks: Integrated Gradients*); Selvaraju et al. (2017 *Grad-CAM*); Adebayo et al. (2018 *Sanity Checks for Saliency Maps*).
- **Purpose:** Implement axiomatic path-integral gradient attribution (Integrated Gradients), conservative relevance backpropagation (LRP), spatial feature activation mapping (Grad-CAM), and Adebayo weight randomization sanity checks.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. XAI Local Surrogates LIME SHAP](01_xai_local_surrogates_lime_shap_and_perturbation_masks.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. XAI Concepts TCAV CBM Counterfactuals](03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Avoid unverified saliency methods (such as Guided Backpropagation) that act as class-agnostic edge filters invariant to model weights; never choose an uninformative zero baseline in Integrated Gradients when zero itself conveys strong semantic signal (causing baseline blindness).

## Mathematical Foundations & Governing Equations

1. **Integrated Gradients Riemann Sum**: $\text{IG}_i(x) = (x_i - x'_i) \times \frac{1}{M} \sum_{m=1}^M \frac{\partial F(x' + \frac{m}{M}(x - x'))}{\partial x_i}$
2. **LRP $\epsilon$-Rule Conservation**: $R_i^{(l)} = \sum_j \frac{z_{ij}}{\sum_{i'} z_{i'j} + \epsilon \cdot \text{sign}(\sum_{i'} z_{i'j})} R_j^{(l+1)}, \quad z_{ij} = a_i w_{ij}$
3. **Grad-CAM Spatial Heatmap**: $\alpha_k^c = \frac{1}{HW}\sum_{u, v} \frac{\partial y^c}{\partial A_{uv}^k}, \quad L_{\text{GradCAM}}^c = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$
4. **Adebayo Randomization Check**: $\text{sim}(S_{\text{orig}}, S_{\text{rand}}) = \frac{S_{\text{orig}} \cdot S_{\text{rand}}}{\|S_{\text{orig}}\|_2 \|S_{\text{rand}}\|_2 + \epsilon}$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_integrated_gradients(x, x_baseline, grads):
    return (x - x_baseline) * grads.mean(dim=0)

def calc_lrp_epsilon(acts, weights_w, rel_next, eps=1e-4):
    z_ij = acts.unsqueeze(-1) * weights_w
    z_j = z_ij.sum(dim=0)
    sign_z = torch.where(z_j >= 0.0, torch.ones_like(z_j), -torch.ones_like(z_j))
    s = rel_next / (z_j + eps * sign_z)
    return (z_ij @ s.unsqueeze(-1)).squeeze(-1)

def calc_grad_cam(acts, grads):
    alpha = grads.mean(dim=(-2, -1))
    cam = (alpha.view(-1, 1, 1) * acts).sum(dim=0)
    return torch.relu(cam)

def calc_cosine_sim(s_orig, s_rand, eps=1e-8):
    dot = (s_orig * s_rand).sum()
    norm_o = torch.sqrt((s_orig * s_orig).sum())
    norm_r = torch.sqrt((s_rand * s_rand).sum())
    return dot / (norm_o * norm_r + eps)

In [ ]:
# Parity verification against golden reference tensors
ig = calc_integrated_gradients(weights["ig_x"], weights["ig_baseline"], weights["ig_grads"])
assert torch.max(torch.abs(ig - weights["ig_ref"])) < 1e-5, "Integrated Gradients mismatch"

lrp = calc_lrp_epsilon(weights["lrp_acts"], weights["lrp_weights"], weights["lrp_rel_next"], eps=1e-4)
assert torch.max(torch.abs(lrp - weights["lrp_ref"])) < 1e-5, "LRP mismatch"

cam = calc_grad_cam(weights["cam_acts"], weights["cam_grads"])
assert torch.max(torch.abs(cam - weights["cam_ref"])) < 1e-5, "Grad-CAM mismatch"

sim = calc_cosine_sim(weights["adebayo_orig"], weights["adebayo_rand"])
assert torch.max(torch.abs(sim - weights["adebayo_sim_ref"])) < 1e-5, "Adebayo similarity mismatch"
print("All IG, LRP, Grad-CAM, and Adebayo assertions passed!")

In [ ]:
# Benchmarking gradient propagation and attribution operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_integrated_gradients(weights["ig_x"], weights["ig_baseline"], weights["ig_grads"])
    _ = calc_lrp_epsilon(weights["lrp_acts"], weights["lrp_weights"], weights["lrp_rel_next"], eps=1e-4)
    _ = calc_grad_cam(weights["cam_acts"], weights["cam_grads"])
    _ = calc_cosine_sim(weights["adebayo_orig"], weights["adebayo_rand"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused IG + LRP + Grad-CAM latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Adebayo Cascading Model Randomization Test
# A faithful attribution collapses towards zero correlation when model weights are randomized
s_learned = weights["adebayo_orig"]
s_randomized = weights["adebayo_rand"]
sim_val = calc_cosine_sim(s_learned, s_randomized).item()
print(f"Attribution alignment under weight randomization: {sim_val:.3f}")
assert sim_val < 0.35, "Faithful attribution must collapse under weight randomization!"
# Degenerate edge detector (unfaithful attribution) maintains high correlation (> 0.9) despite randomized weights
s_degenerate = s_learned + 0.02 * torch.randn_like(s_learned)
sim_degen = calc_cosine_sim(s_learned, s_degenerate).item()
assert sim_degen > 0.95, "Sanity check detects degenerate weight-invariant edge detectors!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Integrated Gradients** satisfies Completeness and Implementation Invariance by integrating along a straight-line interpolation path.
2. **LRP** redistributes relevance conservatively across layers to preserve total evidence.
3. **Grad-CAM** localizes discriminative regions by pooling gradients into channel weights followed by ReLU filtering.
4. **Adebayo Sanity Checks** distinguish genuine decision-logic attributions from weight-invariant edge filters.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.md](../../markdown/10_xai_and_ai_analysis_methods/02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.md)
- **PyTorch 2.x Module:** [02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.py](../../pytorch2/10_xai_and_ai_analysis_methods/02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.py)
- **Functional JAX Module:** [02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.py](../../jax/10_xai_and_ai_analysis_methods/02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.py)
- **Native Rust Candle Track:** [02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.rs](../../candle/10_xai_and_ai_analysis_methods/02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.rs)
- **Hardware Kernel (CPU / WGSL):** [02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.rs](../../kernels/10_xai_and_ai_analysis_methods/02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.rs)
- **Golden SafeTensors:** [02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.safetensors](../../weights/10_xai_and_ai_analysis_methods/02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. XAI Local Surrogates LIME SHAP](01_xai_local_surrogates_lime_shap_and_perturbation_masks.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 03. XAI Concepts TCAV CBM Counterfactuals](03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.ipynb) &rarr;